# Project 6 — Sequence Models (RNN / LSTM / GRU)

## Why sequence models?
Bag-of-Words and TF-IDF **throw away word order**, but order carries meaning:

- *The movie was not bad* → POSITIVE
- *The movie was bad, not good* → NEGATIVE

Same words, different order, opposite sentiment. Sequence models read text **left to right** and remember context.

## Architecture options

| Model | Pros | Cons |
|---|---|---|
| Vanilla RNN | Simple | Vanishing gradients on long sequences |
| **LSTM** | Memory cells + gates → long context | Slower than RNN |
| **GRU** | Smaller, often as good as LSTM | Slightly less expressive |
| Bidirectional LSTM | Reads forwards **and** backwards | 2× parameters |

## What we'll build
An LSTM-based binary sentiment classifier on a small movie-review dataset.

## Step 1 — Imports

In [1]:
import numpy as np, re
import nltk, spacy
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Bidirectional, Dense, Dropout

for pkg in ['stopwords', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

## Step 2 — Sample dataset

In [2]:
texts = [
    'I absolutely loved this movie, the acting was brilliant',
    'What a wonderful film, the storyline was amazing',
    'Best movie I have seen in years, fantastic direction',
    'A masterpiece, every scene was beautifully crafted',
    'Loved every minute of it, the cast was outstanding',
    'Heartwarming story with great performances',
    'An emotional roller coaster that I really enjoyed',
    'Brilliant cinematography and a powerful script',
    'Fun, exciting, and totally worth watching',
    'One of the best films of the decade',
    'Terrible movie, complete waste of two hours',
    'Boring plot and the acting was awful',
    'I hated this film, very disappointing',
    'Worst movie I have seen this year, do not watch',
    'Predictable, dull, and poorly directed',
    'The script was lazy and the characters flat',
    'A complete mess from start to finish',
    'Overrated and forgettable, I want my money back',
    'Awful pacing and a confusing storyline',
    'Just bad — bad acting, bad story, bad everything',
]
labels = np.array([1]*10 + [0]*10)

## Step 3 — Clean with spaCy

In [3]:
def clean(text):
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    doc = nlp(text)
    return ' '.join(tok.lemma_ for tok in doc
                    if not tok.is_stop and not tok.is_space and tok.is_alpha)

cleaned_texts = [clean(t) for t in texts]
print('Original :', texts[0])
print('Cleaned  :', cleaned_texts[0])

Original : I absolutely loved this movie, the acting was brilliant
Cleaned  : absolutely love movie acting brilliant


## Step 4 — Tokenize and pad
An LSTM expects integer sequences of equal length.

In [4]:
VOCAB_SIZE, MAX_LEN, EMBED_DIM, LSTM_UNITS = 200, 10, 16, 16

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(cleaned_texts)
X = pad_sequences(tokenizer.texts_to_sequences(cleaned_texts),
                  maxlen=MAX_LEN, padding='post', truncating='post')

print('Vocab size:', len(tokenizer.word_index))
print('Padded   :', X.shape)
print('First seq:', X[0])

Vocab size: 66
Padded   : (20, 10)
First seq: [17  5  3  6  7  0  0  0  0  0]


## Step 5 — Build the LSTM

Layer-by-layer:
1. **Embedding** — turns word IDs into dense vectors
2. **LSTM** — reads sequence one step at a time, keeping a memory cell
3. **Dropout** — disables random neurons during training (regularization)
4. **Dense(1, sigmoid)** — final probability of positive sentiment

In [5]:
model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    LSTM(16, return_sequences=False),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

d:\Deep_Learning\NLP-Project\venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 6 — Train

In [10]:
history = model.fit(X, labels, epochs=40, verbose=1)
print(f'Final training accuracy: {history.history["accuracy"][-1]:.2f}')

Epoch 1/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9500 - loss: 0.6828
Epoch 2/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.9000 - loss: 0.6821
Epoch 3/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9500 - loss: 0.6800
Epoch 4/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 1.0000 - loss: 0.6799
Epoch 5/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9000 - loss: 0.6785
Epoch 6/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8500 - loss: 0.6806
Epoch 7/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9000 - loss: 0.6768
Epoch 8/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 1.0000 - loss: 0.6751
Epoch 9/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9000 - loss: 0.6751
Epoch 10/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9500 - loss: 0.6683
Epoch 11/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9500 - loss: 0.6628
Epoch 12/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 1.0000 - loss: 0.6636


## Step 7 — Predict on new reviews

In [17]:
new_reviews = [
    'An incredible film with outstanding acting',
    'Boring story and terrible direction, total waste',
    'Not bad — actually enjoyed the second half',
    'I really hated this movie',
]
new_X = pad_sequences(tokenizer.texts_to_sequences([clean(r) for r in new_reviews]),
                     maxlen=MAX_LEN, padding='post')
for r, p in zip(new_reviews, model.predict(new_X, verbose=0)):
    s = 'POSITIVE' if p[0] > 0.5 else 'NEGATIVE'
    print(f'  "{r}" → {s} ({p[0]:.2f})')

  "An incredible film with outstanding acting" → POSITIVE (0.58)
  "Boring story and terrible direction, total waste" → NEGATIVE (0.26)
  "Not bad — actually enjoyed the second half" → NEGATIVE (0.28)
  "I really hated this movie" → NEGATIVE (0.23)


## Bonus — GRU and Bidirectional LSTM

Same problem, different architectures.

In [8]:
gru_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    GRU(LSTM_UNITS),
    Dense(1, activation='sigmoid'),
])
gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
gru_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
bilstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS)),
    Dense(1, activation='sigmoid'),
])
bilstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
bilstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Summary

- Sequence models read text **in order**.
- LSTMs / GRUs use **gates** to remember important earlier words.
- For modern production NLP, replace LSTM with a **Transformer** (BERT, GPT).